# Quality Lab AI Engine - Colab Training

This notebook is only for GPU training on Google Colab.

It does not run FastAPI, does not modify the frontend, and keeps the same `ai-engine/` structure as the local project.

Expected input file in Google Drive:

```text
/content/drive/MyDrive/CGI-FLOW/ai-engine/datasets/raw/tickets.xlsx
```

Final trained model zip will be saved to:

```text
/content/drive/MyDrive/CGI-FLOW/ai-engine/models/supervision-classifier-colab.zip
```

## 1. Mount Google Drive

Mount Drive so the Excel dataset and trained model artifacts can persist outside the Colab VM.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Clone the GitHub Repository

Clone the existing project repository into the Colab runtime. If the folder already exists, pull the latest changes.

In [ ]:
import os
from pathlib import Path

REPO_URL = 'https://github.com/Niima22/CGI-FLOW.git'
REPO_DIR = Path('/content/CGI-FLOW')
AI_ENGINE_DIR = REPO_DIR / 'ai-engine'
DRIVE_PROJECT_DIR = Path('/content/drive/MyDrive/CGI-FLOW')

if REPO_DIR.exists():
    %cd /content/CGI-FLOW
    !git pull
else:
    %cd /content
    !git clone {REPO_URL}

%cd {AI_ENGINE_DIR}

## 3. Install AI Engine Dependencies

Install the dependencies from `ai-engine/requirements.txt`. This includes PyTorch, HuggingFace Transformers, pandas, scikit-learn, and FastAPI dependencies required by the package. FastAPI will not be run in this notebook.

In [ ]:
!pip install -r requirements.txt

## 4. Verify GPU Availability

The classifier can train on CPU, but Colab should be used with GPU for faster training. If GPU is not available, change the Colab runtime type to GPU.

In [ ]:
import torch

gpu_available = torch.cuda.is_available()
print('CUDA available:', gpu_available)

if gpu_available:
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('WARNING: GPU is not available. Training will be much slower. Runtime > Change runtime type > GPU.')

## 5. Read Excel File From Google Drive

Place your Excel file in Google Drive at:

```text
MyDrive/CGI-FLOW/ai-engine/datasets/raw/tickets.xlsx
```

This cell copies it into the cloned Colab repo at `ai-engine/datasets/raw/tickets.xlsx`, keeping the same local folder structure.

In [ ]:
from pathlib import Path
import shutil

drive_raw_dir = DRIVE_PROJECT_DIR / 'ai-engine' / 'datasets' / 'raw'
drive_excel_path = drive_raw_dir / 'tickets.xlsx'
local_raw_dir = AI_ENGINE_DIR / 'datasets' / 'raw'
local_excel_path = local_raw_dir / 'tickets.xlsx'

local_raw_dir.mkdir(parents=True, exist_ok=True)
drive_raw_dir.mkdir(parents=True, exist_ok=True)

if not drive_excel_path.exists():
    raise FileNotFoundError(
        f'Excel file not found: {drive_excel_path}. Upload it to Google Drive first and name it tickets.xlsx.'
    )

shutil.copy2(drive_excel_path, local_excel_path)
print('Excel copied to:', local_excel_path)

## 6. Run Preprocessing

This runs Excel ingestion, NLP text cleaning, missing value handling, invalid row removal, label generation, and train/validation/test split creation.

In [ ]:
!python -m preprocessing.excel_ingestion --excel datasets/raw/tickets.xlsx --sheet Picking

!ls -lh datasets/processed

## 7. Train the Supervision Classifier With GPU

Train CamemBERT for multi-label classification. The model predicts the three quality criteria separately:

- `synthese_demande`
- `actions_resultat`
- `formule_politesse`

The final conformity is computed by the business rule in the inference layer.

In [ ]:
!python -m training.train_classifier \
  --dataset-dir datasets/processed \
  --output-dir models/supervision-classifier-colab \
  --base-model camembert-base \
  --epochs 1 \
  --batch-size 8

## 8. Verify Saved Model Artifacts

Check that the trained model, tokenizer, config, and evaluation output exist. This folder can be copied locally into `ai-engine/models/`.

In [ ]:
from pathlib import Path

model_dir = Path('models/supervision-classifier-colab')
required_files = ['config.json', 'model.safetensors']

print('Model directory:', model_dir.resolve())
print('Exists:', model_dir.exists())
!find models/supervision-classifier-colab -maxdepth 2 -type f | sort | head -50

missing = [name for name in required_files if not (model_dir / name).exists()]
if missing:
    raise FileNotFoundError(f'Missing expected model files: {missing}')

print('Model artifacts verified.')

## 9. Zip the Trained Model Folder

Create `models/supervision-classifier-colab.zip` so it can be downloaded or copied back into your local project.

In [ ]:
!cd models && zip -r supervision-classifier-colab.zip supervision-classifier-colab
!ls -lh models/supervision-classifier-colab.zip

## 10. Save the Trained Model Zip to Google Drive

Copy the zip to Google Drive so you can download it later and place it locally in `ai-engine/models/`.

In [ ]:
drive_models_dir = DRIVE_PROJECT_DIR / 'ai-engine' / 'models'
drive_models_dir.mkdir(parents=True, exist_ok=True)

local_zip = AI_ENGINE_DIR / 'models' / 'supervision-classifier-colab.zip'
drive_zip = drive_models_dir / 'supervision-classifier-colab.zip'

shutil.copy2(local_zip, drive_zip)
print('Saved trained model zip to:', drive_zip)

## 11. Use the Model Locally

After downloading `supervision-classifier-colab.zip`, extract it into your local project as:

```text
ai-engine/models/supervision-classifier/
```

or keep the Colab name and pass the folder explicitly when loading the inference class.

The local FastAPI inference endpoint `/analyze-ticket` expects a trained classifier folder under `ai-engine/models/supervision-classifier/` by default.